In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

dataset_path = "/kaggle/working/shakespeare.txt"

urllib.request.urlretrieve(url, dataset_path)

print("Dataset downloaded successfully!")
print("Location:", dataset_path)

In [ ]:
with open(dataset_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Total number of characters:", len(text))

print("\nFirst 1000 characters:\n")
print(text[:1000])

In [ ]:
print("Total characters:", len(text))

print("Number of unique characters:", len(set(text)))

print("\nUnique characters:")
print(sorted(set(text)))

In [ ]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocabulary size:", vocab_size)

char_to_idx = {char: idx for idx, char in enumerate(chars)}

idx_to_char = {idx: char for idx, char in enumerate(chars)}

print("\nCharacter to index:")
print(list(char_to_idx.items())[:20])

In [ ]:
encoded_text = np.array(
    [char_to_idx[c] for c in text],
    dtype=np.int64
)

print("Encoded shape:", encoded_text.shape)

print("\nFirst 50 encoded characters:")
print(encoded_text[:50])

In [ ]:
split_index = int(0.9 * len(encoded_text))

train_data = encoded_text[:split_index]
val_data = encoded_text[split_index:]

print("Training characters:", len(train_data))
print("Validation characters:", len(val_data))

In [ ]:
class ShakespeareDataset(Dataset):

    def __init__(self, data, seq_length=100):
        self.data = torch.tensor(data, dtype=torch.long)
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, index):

        x = self.data[index:index + self.seq_length]

        y = self.data[index + 1:index + self.seq_length + 1]

        return x, y

In [ ]:
SEQ_LENGTH = 100
BATCH_SIZE = 128

train_dataset = ShakespeareDataset(
    train_data,
    seq_length=SEQ_LENGTH
)

val_dataset = ShakespeareDataset(
    val_data,
    seq_length=SEQ_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
x_batch, y_batch = next(iter(train_loader))

print("Input shape :", x_batch.shape)
print("Target shape:", y_batch.shape)

print("\nFirst input sequence:")
print("".join(idx_to_char[int(i)] for i in x_batch[0]))

print("\nCorresponding target:")
print("".join(idx_to_char[int(i)] for i in y_batch[0]))

In [ ]:
class CharLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=256,
        num_layers=1,
        dropout=0.0
    ):

        super(CharLSTM, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x, hidden=None):

        x = self.embedding(x)

        output, hidden = self.lstm(
            x,
            hidden
        )

        output = self.fc(output)

        return output, hidden

In [ ]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

model = CharLSTM(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

model = model.to(device)

print(model)

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.002
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output, _ = model(x)

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            y.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            output, _ = model(x)

            loss = criterion(
                output.reshape(-1, output.size(-1)),
                y.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 10

train_losses = []
val_losses = []

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss = validate(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_char_lstm.pth"
        )

        print("Best model saved!")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    train_losses,
    label="Training Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.grid()

plt.show()

In [ ]:
import math

train_perplexity = math.exp(train_losses[-1])
val_perplexity = math.exp(val_losses[-1])

print("Final Training Loss:", train_losses[-1])
print("Final Validation Loss:", val_losses[-1])

print("Training Perplexity:", train_perplexity)
print("Validation Perplexity:", val_perplexity)

In [ ]:
model.load_state_dict(
    torch.load(
        "/kaggle/working/best_char_lstm.pth",
        map_location=device
    )
)

model.eval()

print("Best model loaded successfully.")

In [ ]:
def generate_text(
    model,
    start_string,
    length=200,
    temperature=0.8
):

    model.eval()

    # Make sure all characters exist
    for char in start_string:
        if char not in char_to_idx:
            raise ValueError(
                f"Character {repr(char)} "
                f"is not present in the vocabulary."
            )

    input_indices = [
        char_to_idx[c]
        for c in start_string
    ]

    input_tensor = torch.tensor(
        [input_indices],
        dtype=torch.long
    ).to(device)

    generated_text = start_string

    hidden = None

    with torch.no_grad():

        output, hidden = model(
            input_tensor,
            hidden
        )

        last_input = input_tensor[:, -1:]

        for _ in range(length):

            output, hidden = model(
                last_input,
                hidden
            )

            logits = output[:, -1, :]

            logits = logits / temperature

            probabilities = torch.softmax(
                logits,
                dim=-1
            )

            next_index = torch.multinomial(
                probabilities,
                num_samples=1
            )

            next_char = idx_to_char[
                next_index.item()
            ]

            generated_text += next_char

            last_input = next_index

    return generated_text

In [ ]:
generated = generate_text(
    model,
    start_string="ROMEO",
    length=300,
    temperature=0.8
)

print(generated)

In [ ]:
start_words = [
    "ROMEO",
    "JULIET",
    "KING",
    "What",
    "The"
]

for word in start_words:

    print("=" * 80)
    print("START:", word)
    print()

    generated = generate_text(
        model,
        start_string=word,
        length=200,
        temperature=0.8
    )

    print(generated)
    print()

In [ ]:
temperatures = [0.3, 0.7, 1.0, 1.3]

for temp in temperatures:

    print("=" * 80)
    print("Temperature:", temp)
    print()

    generated = generate_text(
        model,
        start_string="ROMEO",
        length=250,
        temperature=temp
    )

    print(generated)

In [ ]:
import json

with open(
    "/kaggle/working/char_to_idx.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        char_to_idx,
        f,
        ensure_ascii=False
    )

with open(
    "/kaggle/working/idx_to_char.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {str(k): v for k, v in idx_to_char.items()},
        f,
        ensure_ascii=False
    )

print("Vocabulary files saved.")

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab_size": vocab_size,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "char_to_idx": char_to_idx,
    "idx_to_char": idx_to_char
}

torch.save(
    checkpoint,
    "/kaggle/working/shakespeare_lstm_checkpoint.pth"
)

print("Complete checkpoint saved!")

In [ ]:
print("=" * 60)
print("TEXT GENERATION USING LSTM")
print("=" * 60)

print("Dataset: Tiny Shakespeare")
print("Task: Character-level text generation")
print("Vocabulary size:", vocab_size)
print("Sequence length:", SEQ_LENGTH)
print("Embedding dimension:", EMBEDDING_DIM)
print("Hidden dimension:", HIDDEN_DIM)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

print("\nFinal Training Loss:", train_losses[-1])
print("Final Validation Loss:", val_losses[-1])
print("Training Perplexity:", train_perplexity)
print("Validation Perplexity:", val_perplexity)

print("\nBest model saved at:")
print("/kaggle/working/best_char_lstm.pth")